## structured concurrency og kansellering

Coroutines bruker `structured concurrency` som betyr at en coroutine kan ikke kjøre utenfor en annen coroutine. Dette gjør det enklere å håndtere livssyklusen til coroutines og sørger for at de blir kansellert når de ikke lenger er nødvendige. En foreldre-coroutine kan starte flere barne-coroutines.

Når foreldre-coroutinen blir kansellert, blir alle barne-coroutines også kansellert. Dette gjør at man slipper å bekymre seg for coroutines som henger igjen og uventet fortsetter å kjøre. Dersom en barne-coroutine kaster en unntak, vil det også føre til at foreldre-coroutinen og alle andre barne-coroutines blir kansellert. Dette sikrer at feil i en del av programmet ikke fører til at andre deler fortsetter å kjøre i en uforutsigbar tilstand.

`Cooperative cancellation` betyr at en coroutine må aktivt sjekke om den har blitt kansellert, og respondere på det ved å avslutte sin utførelse. Denne sjekken gjøres i `suspension points` som `delay()`, `yield()`, eller ved å sjekke `isActive`-flagget. Dette gir utvikleren kontroll over når og hvordan en coroutine skal avslutte, og gjør det mulig å utføre nødvendige oppryddingsoperasjoner før coroutinen termineres.

In [ ]:
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.delay
import kotlinx.coroutines.isActive
import kotlinx.coroutines.launch
import kotlinx.coroutines.runBlocking
import kotlinx.coroutines.withContext
import kotlinx.coroutines.yield

class MyRunnerUncancellable() {

    suspend fun run() = withContext(Dispatchers.Default) {
        println("Running...")
        val job = launch {
            listen()
            println("Launched listening job...")
        }

        delay(1.seconds)
        job.cancel()
    }


    private suspend fun listen() = coroutineScope {
        while(isActive) {
            try {
                println("Listening...")
                doSomething()
                delay(10.seconds) // Suspension point
            } catch (e: Exception) {
                println("Error listening: ${e.message}")
            }
        }
    }

    private suspend fun doSomething() {
        repeat(10) {
            try {
                println("doing something...")
                delay(1.seconds)
            } catch (e: Exception) {
                println("Swallowed exception of type : ${e::class.qualifiedName}")
                println("Error doing something: ${e.message}")
            }
        }
    }
}

runBlocking {
    MyRunnerUncancellable().run()
}


In [ ]:
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.delay
import kotlinx.coroutines.isActive
import kotlinx.coroutines.launch
import kotlinx.coroutines.runBlocking
import kotlinx.coroutines.withContext
import kotlin.coroutines.cancellation.CancellationException
import kotlin.time.Duration

class MyRunnerCancellable() {

    suspend fun run() = withContext(Dispatchers.Default) {
        println("Running...")
        val job = launch {
            listen()
            println("Launched listening job...")
        }

        delay(1.seconds)
        job.cancel()
    }


    private suspend fun listen() = coroutineScope {
        while(isActive) {
            try {
                println("Listening cancellable...")
                launch {
                    doSomething()
                }
                delay(10.seconds)
            }catch (e: CancellationException) {
                println("Error listening: ${e.message}")
            }
        }
    }

    private suspend fun doSomething() {
            try {
                println("doing something...")
                delay(Duration.INFINITE)
            } catch (e: CancellationException) {
                println("Error doing something: ${e.message}")
                throw e
            }
    }
}

runBlocking {
    MyRunnerCancellable().run()
}

In [ ]:
import kotlinx.coroutines.CoroutineExceptionHandler
import kotlinx.coroutines.CoroutineName
import kotlinx.coroutines.CoroutineScope
import kotlinx.coroutines.SupervisorJob

runBlocking {
    supervisor()
}

suspend fun supervisor() {
    // En supervisor lar coroutinene gjør livssyklusen til coroutines individuell, hvis et barn kanselleres eller feiler
    // påvirker ikke dette foreldre eller søsken
    val supervisor = SupervisorJob()
    // Man kan definere egne exception handlers for en coroutine
    val handler = CoroutineExceptionHandler { context, ex ->
        println("Exception handler caught: ${ex::class.qualifiedName}, context: ${context.get(CoroutineName)}")
    }
    // Man kan plusse sammmen ulike elementer til et custom scope
    val customScope = CoroutineScope(Dispatchers.IO + supervisor + CoroutineName("CustomScope"))

    val firstChild = customScope.launch(handler) { // Hvert barn trenger sin egen exception handler ved bruk av supervisors
        println("The first child is failing")
        throw AssertionError("The first child is cancelled")
    }

    val secondChild = customScope.launch {
        firstChild.join()
        // Cancellation of the first child is not propagated to the second child
        println("The first child is cancelled: ${firstChild.isCancelled}, but the second one is still active")
        try {
            delay(Long.MAX_VALUE)
        } finally {
            // But cancellation of the supervisor is propagated
            println("The second child is cancelled because the supervisor was cancelled")
        }
    }
    // wait until the first child fails & completes
    firstChild.join()
    println("Cancelling the supervisor")
    supervisor.cancel()
    secondChild.join()
}